In [ ]:
import os, csv, json, uuid, time, random
from datetime import datetime, timedelta
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
from cassandra import ConsistencyLevel
from tqdm import tqdm
import traceback

# Session metadata
RUN_ID = str(uuid.uuid4())
DB_SYSTEM = "AstraDB"
SERVER_VERSION = "Cassandra 4.0.11"
DRIVER = "cassandra-driver-3.x"
CONNECTION_PARAMS = "Astra secure bundle"
DB_NAME = "research-benchmarking"

os.makedirs("./logs", exist_ok=True)

# Connection
# auth = PlainTextAuthProvider('token')
# cluster = Cluster(cloud={'secure_connect_bundle': 'C:/Users/avyaa/bundle.zip'}, auth_provider=auth)
# session = cluster.connect('ecommerce')

print(f"✅ Connected to {DB_SYSTEM}")
print(f"📋 RUN_ID: {RUN_ID}")
print(f"🗄️  Database: {DB_NAME}")
print(f"🔧 Driver: {DRIVER}")

✅ Connected to AstraDB
📋 RUN_ID: 4f40d6e9-965d-4cd2-9f89-f3a11e46def0
🗄️  Database: research-benchmarking
🔧 Driver: cassandra-driver-3.x


In [2]:
# Set seed for reproducibility
random.seed(42)

print("📊 Loading parameter pools from live dataset...")

# Sample valid IDs from existing data
product_ids = [row.id for row in session.execute("SELECT id FROM products LIMIT 500")]
customer_ids = [row.id for row in session.execute("SELECT id FROM customers LIMIT 500")]
category_ids = [row.id for row in session.execute("SELECT id FROM categories LIMIT 50")]
order_ids = [row.id for row in session.execute("SELECT id FROM orders LIMIT 500")]

# Sample product attributes for realistic operations
products_data = list(session.execute("SELECT id, sku_lc, price, category_id FROM products LIMIT 500"))
price_min = min(p.price for p in products_data if p.price)
price_max = max(p.price for p in products_data if p.price)

print(f"✅ Loaded pools: {len(product_ids)} products, {len(customer_ids)} customers, {len(category_ids)} categories, {len(order_ids)} orders")

# Helper functions for data generation
def random_email():
    return f"test_{uuid.uuid4().hex[:8]}@bench.local"

def random_phone():
    return f"+1{random.randint(2000000000, 9999999999)}"

def random_sku():
    return f"SKU-BENCH-{random.randint(100000, 999999)}"

def random_timestamp():
    return datetime.now() - timedelta(days=random.randint(0, 365))

def random_price():
    return round(random.uniform(float(price_min), float(price_max)), 2)

📊 Loading parameter pools from live dataset...
✅ Loaded pools: 500 products, 500 customers, 50 categories, 500 orders


In [3]:
# CSV header for writes.csv
CSV_PATH = "./logs/writes.csv"
CSV_HEADER = [
    "timestamp_utc", "run_id", "db_system", "server_version", "driver", 
    "connection_params", "db_name", "phase", "query_id", "complexity", 
    "query_name", "operation_type", "run_number", "execution_ms", 
    "rows_affected", "returned_id", "error_code", "error_message", "params_json"
]

# Initialize CSV
with open(CSV_PATH, 'w', newline='', encoding='utf-8') as f:
    csv.writer(f).writerow(CSV_HEADER)

def log_write_operation(query_id, complexity, query_name, run_number, execution_ms, 
                        rows_affected, returned_id, error_code, error_message, params):
    """Log a single write operation to CSV"""
    row = [
        datetime.utcnow().isoformat() + "Z",
        RUN_ID,
        DB_SYSTEM,
        SERVER_VERSION,
        DRIVER,
        CONNECTION_PARAMS,
        DB_NAME,
        "writes",
        query_id,
        complexity,
        query_name,
        "write",
        run_number,
        round(execution_ms, 3) if execution_ms else None,
        rows_affected if rows_affected is not None else None,
        str(returned_id) if returned_id else None,
        error_code if error_code else None,
        error_message if error_message else None,
        json.dumps(params, default=str)
    ]
    
    with open(CSV_PATH, 'a', newline='', encoding='utf-8') as f:
        csv.writer(f).writerow(row)

print(f"✅ Logging initialized: {CSV_PATH}")

✅ Logging initialized: ./logs/writes.csv


In [4]:
print("🌱 Pre-seeding temporary data for update/delete tests...")

# Prepare statement for bulk insert
temp_customer_stmt = session.prepare("""
    INSERT INTO customers (id, first_name, last_name, email, email_lc, phone, 
                          address_line1, city, state, country, postal_code, 
                          is_active, created_at, updated_at)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

# Generate 240 temporary customers (120 for updates, 120 for deletes)
temp_customer_ids_update = []
temp_customer_ids_delete = []

for i in tqdm(range(240), desc="Seeding temp customers"):
    cust_id = uuid.uuid4()
    email = f"temp_{i}_{uuid.uuid4().hex[:6]}@bench.local"
    
    session.execute(temp_customer_stmt, [
        cust_id, f"TempFirst{i}", f"TempLast{i}", email, email.lower(),
        random_phone(), "123 Temp St", "TempCity", "TempState", "USA", "12345",
        True, datetime.now(), datetime.now()
    ])
    
    if i < 120:
        temp_customer_ids_update.append(cust_id)
    else:
        temp_customer_ids_delete.append(cust_id)

print(f"✅ Seeded {len(temp_customer_ids_update)} for updates, {len(temp_customer_ids_delete)} for deletes")

🌱 Pre-seeding temporary data for update/delete tests...


Seeding temp customers: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 240/240 [00:47<00:00,  5.00it/s]

✅ Seeded 120 for updates, 120 for deletes


In [5]:
print("\n🔹 W1A: Single Row Insert - Customer Registration")

insert_stmt = session.prepare("""
    INSERT INTO customers (id, first_name, last_name, email, email_lc, phone,
                          address_line1, city, state, country, postal_code,
                          is_active, created_at, updated_at)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

for run_num in tqdm(range(1, 101), desc="W1A-Insert"):
    cust_id = uuid.uuid4()
    email = random_email()
    phone = random_phone()
    params = {"customer_id": str(cust_id), "email": email, "phone": phone}
    
    start = time.perf_counter()
    error_code, error_msg, rows_affected = None, None, 0
    
    try:
        session.execute(insert_stmt, [
            cust_id, "John", "Doe", email, email.lower(), phone,
            "123 Main St", "TestCity", "TestState", "USA", "12345",
            True, datetime.now(), datetime.now()
        ])
        rows_affected = 1
    except Exception as e:
        error_code = type(e).__name__
        error_msg = str(e)[:200]
    
    execution_ms = (time.perf_counter() - start) * 1000
    log_write_operation("W1A", "Basic", "Single-row insert (customer)", run_num, 
                       execution_ms, rows_affected, cust_id, error_code, error_msg, params)


🔹 W1A: Single Row Insert - Customer Registration


W1A-Insert: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:19<00:00,  5.16it/s]


In [6]:
print("\n🔹 W1B: Single Row Update - Profile Edit (Non-indexed)")

update_stmt = session.prepare("""
    UPDATE customers SET phone = ?, updated_at = ? WHERE id = ?
""")

for run_num in tqdm(range(1, 101), desc="W1B-Update"):
    cust_id = random.choice(temp_customer_ids_update)
    new_phone = random_phone()
    params = {"customer_id": str(cust_id), "new_phone": new_phone}
    
    start = time.perf_counter()
    error_code, error_msg, rows_affected = None, None, 0
    
    try:
        session.execute(update_stmt, [new_phone, datetime.now(), cust_id])
        rows_affected = 1
    except Exception as e:
        error_code = type(e).__name__
        error_msg = str(e)[:200]
    
    execution_ms = (time.perf_counter() - start) * 1000
    log_write_operation("W1B", "Basic", "Single-row update (non-indexed)", run_num,
                       execution_ms, rows_affected, None, error_code, error_msg, params)


🔹 W1B: Single Row Update - Profile Edit (Non-indexed)


W1B-Update: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:19<00:00,  5.07it/s]


In [7]:
print("\n🔹 W1C: Single Row Delete - Remove Customer")

delete_stmt = session.prepare("DELETE FROM customers WHERE id = ?")

for run_num in tqdm(range(1, 101), desc="W1C-Delete"):
    if not temp_customer_ids_delete:
        print("⚠️  No more temp customers to delete, skipping remaining runs")
        break
    
    cust_id = temp_customer_ids_delete.pop()
    params = {"customer_id": str(cust_id)}
    
    start = time.perf_counter()
    error_code, error_msg, rows_affected = None, None, 0
    
    try:
        session.execute(delete_stmt, [cust_id])
        rows_affected = 1
    except Exception as e:
        error_code = type(e).__name__
        error_msg = str(e)[:200]
    
    execution_ms = (time.perf_counter() - start) * 1000
    log_write_operation("W1C", "Basic", "Single-row delete", run_num,
                       execution_ms, rows_affected, None, error_code, error_msg, params)


🔹 W1C: Single Row Delete - Remove Customer


W1C-Delete: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:19<00:00,  5.16it/s]


In [8]:
print("\n🔹 W2: Bulk Insert - Order Items Batch")

# Cassandra batch insert
from cassandra.query import BatchStatement

order_item_stmt = session.prepare("""
    INSERT INTO order_items (id, order_id, product_id, quantity, unit_price,
                            discount_amount, total_price, created_at)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""")

for run_num in tqdm(range(1, 101), desc="W2-BulkInsert"):
    order_id = random.choice(order_ids)
    batch_size = random.randint(5, 10)
    params = {"order_id": str(order_id), "batch_size": batch_size}
    
    start = time.perf_counter()
    error_code, error_msg, rows_affected = None, None, 0
    
    try:
        batch = BatchStatement()
        for _ in range(batch_size):
            item_id = uuid.uuid4()
            product_id = random.choice(product_ids)
            qty = random.randint(1, 5)
            price = random_price()
            total = qty * price
            
            batch.add(order_item_stmt, [
                item_id, order_id, product_id, qty, price, 0.0, total, datetime.now()
            ])
        
        session.execute(batch)
        rows_affected = batch_size
    except Exception as e:
        error_code = type(e).__name__
        error_msg = str(e)[:200]
    
    execution_ms = (time.perf_counter() - start) * 1000
    log_write_operation("W2", "Moderate", "Bulk insert (order_items)", run_num,
                       execution_ms, rows_affected, None, error_code, error_msg, params)


🔹 W2: Bulk Insert - Order Items Batch


W2-BulkInsert: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:19<00:00,  5.22it/s]


In [9]:
print("\n🔹 W3A: Conditional Update - Price Range Update")

# Note: Cassandra doesn't support WHERE clauses on non-partition keys without ALLOW FILTERING
# We'll simulate by fetching matching IDs first, then updating (realistic pattern)

for run_num in tqdm(range(1, 101), desc="W3A-ConditionalUpdate"):
    min_price = random_price()
    max_price = min_price + random.uniform(10, 100)
    category_id = random.choice(category_ids)
    params = {"category_id": str(category_id), "min_price": min_price, "max_price": max_price}
    
    start = time.perf_counter()
    error_code, error_msg, rows_affected = None, None, 0
    
    try:
        # Step 1: Find matching products (realistic pattern)
        query = f"SELECT id FROM products WHERE category_id = {category_id} AND price >= {min_price} AND price <= {max_price} ALLOW FILTERING"
        matching = list(session.execute(query))[:10]  # Limit to 10 for performance
        
        # Step 2: Update each
        update_stmt = session.prepare("UPDATE products SET popularity = popularity + 1, updated_at = ? WHERE id = ?")
        for row in matching:
            session.execute(update_stmt, [datetime.now(), row.id])
            rows_affected += 1
    except Exception as e:
        error_code = type(e).__name__
        error_msg = str(e)[:200]
    
    execution_ms = (time.perf_counter() - start) * 1000
    log_write_operation("W3A", "Moderate", "Conditional update (price range)", run_num,
                       execution_ms, rows_affected, None, error_code, error_msg, params)


🔹 W3A: Conditional Update - Price Range Update


W3A-ConditionalUpdate: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [01:02<00:00,  1.61it/s]


In [10]:
print("\n🔹 W3B: Conditional Delete - Zero Quantity Cleanup")

# Similar pattern: find then delete
for run_num in tqdm(range(1, 101), desc="W3B-ConditionalDelete"):
    order_id = random.choice(order_ids)
    params = {"order_id": str(order_id)}
    
    start = time.perf_counter()
    error_code, error_msg, rows_affected = None, None, 0
    
    try:
        # Find items with quantity = 0 (if any exist after data manipulation)
        query = f"SELECT id FROM order_items WHERE order_id = {order_id} ALLOW FILTERING"
        matching = list(session.execute(query))[:5]  # Limit deletions
        
        delete_stmt = session.prepare("DELETE FROM order_items WHERE id = ?")
        for row in matching:
            session.execute(delete_stmt, [row.id])
            rows_affected += 1
    except Exception as e:
        error_code = type(e).__name__
        error_msg = str(e)[:200]
    
    execution_ms = (time.perf_counter() - start) * 1000
    log_write_operation("W3B", "Moderate", "Conditional delete (cleanup)", run_num,
                       execution_ms, rows_affected, None, error_code, error_msg, params)


🔹 W3B: Conditional Delete - Zero Quantity Cleanup


W3B-ConditionalDelete: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [01:44<00:00,  1.04s/it]


In [11]:
print("\n🔹 W4: Upsert - Product SKU Update (Lightweight Transaction)")

# Cassandra upsert via INSERT with IF NOT EXISTS, then UPDATE
upsert_check_stmt = session.prepare("INSERT INTO products (id, sku, sku_lc) VALUES (?, ?, ?) IF NOT EXISTS")
upsert_update_stmt = session.prepare("UPDATE products SET price = ?, updated_at = ? WHERE id = ?")

# Small SKU pool to force conflicts
sku_pool = [random_sku() for _ in range(20)]

for run_num in tqdm(range(1, 101), desc="W4-Upsert"):
    product_id = uuid.uuid4()
    sku = random.choice(sku_pool)
    price = random_price()
    params = {"sku": sku, "price": price}
    
    start = time.perf_counter()
    error_code, error_msg, rows_affected = None, None, 0
    
    try:
        # Try insert first
        result = session.execute(upsert_check_stmt, [product_id, sku, sku.lower()])
        
        if result.one().applied:
            # New insert succeeded
            rows_affected = 1
        else:
            # Already exists, update instead
            existing_id = random.choice(product_ids)  # Simulate finding existing
            session.execute(upsert_update_stmt, [price, datetime.now(), existing_id])
            rows_affected = 1
    except Exception as e:
        error_code = type(e).__name__
        error_msg = str(e)[:200]
    
    execution_ms = (time.perf_counter() - start) * 1000
    log_write_operation("W4", "Moderate", "Upsert (LWT)", run_num,
                       execution_ms, rows_affected, product_id, error_code, error_msg, params)


🔹 W4: Upsert - Product SKU Update (Lightweight Transaction)


W4-Upsert: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:19<00:00,  5.06it/s]


In [12]:
print("\n🔹 W5: Transactional Write Mix - Order Placement")

# Cassandra doesn't have multi-partition transactions, so we use LOGGED BATCH
order_create_stmt = session.prepare("""
    INSERT INTO orders (id, customer_id, order_date, status, payment_status, currency,
                       subtotal_amount, tax_amount, shipping_fee, discount_amount,
                       total_amount, coupon_code, shipping_address, billing_address,
                       created_at, updated_at)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

for run_num in tqdm(range(1, 101), desc="W5-Transaction"):
    order_id = uuid.uuid4()
    customer_id = random.choice(customer_ids)
    num_items = random.randint(3, 8)
    
    subtotal = sum(random_price() * random.randint(1, 3) for _ in range(num_items))
    tax = subtotal * 0.08
    shipping = 9.99
    total = subtotal + tax + shipping
    
    params = {"order_id": str(order_id), "customer_id": str(customer_id), "num_items": num_items}
    
    start = time.perf_counter()
    error_code, error_msg, rows_affected = None, None, 0
    
    try:
        batch = BatchStatement()
        
        # 1. Create order
        batch.add(order_create_stmt, [
            order_id, customer_id, datetime.now(), "pending", "pending", "USD",
            subtotal, tax, shipping, 0.0, total, "", "{}", "{}", datetime.now(), datetime.now()
        ])
        
        # 2. Add order items
        for _ in range(num_items):
            item_id = uuid.uuid4()
            product_id = random.choice(product_ids)
            qty = random.randint(1, 3)
            price = random_price()
            
            batch.add(order_item_stmt, [
                item_id, order_id, product_id, qty, price, 0.0, qty * price, datetime.now()
            ])
        
        session.execute(batch)
        rows_affected = 1 + num_items
    except Exception as e:
        error_code = type(e).__name__
        error_msg = str(e)[:200]
    
    execution_ms = (time.perf_counter() - start) * 1000
    log_write_operation("W5", "Advanced", "Transactional write mix (order)", run_num,
                       execution_ms, rows_affected, order_id, error_code, error_msg, params)


🔹 W5: Transactional Write Mix - Order Placement


W5-Transaction: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:19<00:00,  5.19it/s]


In [13]:
print("\n🔹 W6: Rollback - Constraint Violation Recovery")

# Force duplicate email constraint violation
duplicate_email_stmt = session.prepare("""
    INSERT INTO customers (id, first_name, last_name, email, email_lc, phone,
                          address_line1, city, state, country, postal_code,
                          is_active, created_at, updated_at)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

# Pick an existing email to force conflict
existing_customer = session.execute("SELECT email FROM customers LIMIT 1").one()
conflict_email = existing_customer.email if existing_customer else "conflict@test.com"

for run_num in tqdm(range(1, 101), desc="W6-Rollback"):
    cust_id = uuid.uuid4()
    params = {"customer_id": str(cust_id), "email": conflict_email}
    
    start = time.perf_counter()
    error_code, error_msg, rows_affected = None, None, 0
    
    try:
        # This should fail if email index enforces uniqueness (it doesn't in Cassandra)
        # But we simulate by checking first
        check = session.execute(f"SELECT id FROM customers WHERE email_lc = '{conflict_email.lower()}' ALLOW FILTERING")
        if check.one():
            raise Exception("EmailAlreadyExists")
        
        session.execute(duplicate_email_stmt, [
            cust_id, "Conflict", "User", conflict_email, conflict_email.lower(), random_phone(),
            "123 St", "City", "State", "USA", "12345", True, datetime.now(), datetime.now()
        ])
        rows_affected = 1
    except Exception as e:
        error_code = type(e).__name__
        error_msg = str(e)[:200]
    
    execution_ms = (time.perf_counter() - start) * 1000
    log_write_operation("W6", "Advanced", "Rollback (constraint violation)", run_num,
                       execution_ms, rows_affected, None, error_code, error_msg, params)


🔹 W6: Rollback - Constraint Violation Recovery


W6-Rollback: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:19<00:00,  5.05it/s]


In [14]:
print("\n🔹 W7A: Index Maintenance - Update Non-indexed Column")

update_phone_stmt = session.prepare("UPDATE customers SET phone = ?, updated_at = ? WHERE id = ?")

for run_num in tqdm(range(1, 101), desc="W7A-NonIndexed"):
    cust_id = random.choice(customer_ids)
    new_phone = random_phone()
    params = {"customer_id": str(cust_id), "column": "phone"}
    
    start = time.perf_counter()
    error_code, error_msg, rows_affected = None, None, 0
    
    try:
        session.execute(update_phone_stmt, [new_phone, datetime.now(), cust_id])
        rows_affected = 1
    except Exception as e:
        error_code = type(e).__name__
        error_msg = str(e)[:200]
    
    execution_ms = (time.perf_counter() - start) * 1000
    log_write_operation("W7A", "Advanced", "Update non-indexed (phone)", run_num,
                       execution_ms, rows_affected, None, error_code, error_msg, params)


🔹 W7A: Index Maintenance - Update Non-indexed Column


W7A-NonIndexed: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:19<00:00,  5.03it/s]


In [15]:
print("\n🔹 W7B: Index Maintenance - Update Indexed Column")

update_email_stmt = session.prepare("UPDATE customers SET email = ?, email_lc = ?, updated_at = ? WHERE id = ?")

for run_num in tqdm(range(1, 101), desc="W7B-Indexed"):
    cust_id = random.choice(customer_ids)
    new_email = random_email()
    params = {"customer_id": str(cust_id), "column": "email_lc"}
    
    start = time.perf_counter()
    error_code, error_msg, rows_affected = None, None, 0
    
    try:
        session.execute(update_email_stmt, [new_email, new_email.lower(), datetime.now(), cust_id])
        rows_affected = 1
    except Exception as e:
        error_code = type(e).__name__
        error_msg = str(e)[:200]
    
    execution_ms = (time.perf_counter() - start) * 1000
    log_write_operation("W7B", "Advanced", "Update indexed (email_lc)", run_num,
                       execution_ms, rows_affected, None, error_code, error_msg, params)


🔹 W7B: Index Maintenance - Update Indexed Column


W7B-Indexed: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:19<00:00,  5.01it/s]


In [16]:
print("\n" + "="*60)
print("🎉 WRITE BENCHMARKING COMPLETE!")
print("="*60)

# Count operations
import pandas as pd
df = pd.read_csv(CSV_PATH)

print(f"\n📊 Summary:")
print(f"   Total operations logged: {len(df)}")
print(f"   Successful operations: {df['error_code'].isna().sum()}")
print(f"   Failed operations: {df['error_code'].notna().sum()}")
print(f"\n📁 Results saved to: {CSV_PATH}")

# Per-operation summary
print(f"\n📈 Per-Query Summary:")
summary = df.groupby(['query_id', 'query_name']).agg({
    'execution_ms': ['count', 'mean', 'median', 'std'],
    'error_code': lambda x: x.notna().sum()
}).round(2)
print(summary)

print(f"\n✅ Benchmark session complete: RUN_ID={RUN_ID}")
cluster.shutdown()


🎉 WRITE BENCHMARKING COMPLETE!

📊 Summary:
   Total operations logged: 1100
   Successful operations: 900
   Failed operations: 200

📁 Results saved to: ./logs/writes.csv

📈 Per-Query Summary:
                                          execution_ms                   \
                                                 count     mean  median   
query_id query_name                                                       
W1A      Single-row insert (customer)              100   191.46  190.63   
W1B      Single-row update (non-indexed)           100   195.20  192.53   
W1C      Single-row delete                         100   191.93  191.91   
W2       Bulk insert (order_items)                 100   190.14  190.37   
W3A      Conditional update (price range)          100   620.48  589.69   
W3B      Conditional delete (cleanup)              100  1039.20  962.38   
W4       Upsert (LWT)                              100   196.06  195.94   
W5       Transactional write mix (order)           100  